# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [28]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score,
    precision_recall_curve, f1_score
)

from imblearn.over_sampling import RandomOverSampler,SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [29]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

,distance_from_home,distance_from_last_transaction,ratio_to_median_purchase_price,repeat_retailer,used_chip,used_pin_number,online_order,fraud
0,57.877857,0.311140,1.945940,1.0,1.0,0.0,0.0,0.0
1,10.829943,0.175592,1.294219,1.0,0.0,0.0,0.0,0.0
2,5.091079,0.805153,0.427715,1.0,0.0,0.0,1.0,0.0
3,2.247564,5.600044,0.362663,1.0,1.0,0.0,1.0,0.0
4,44.190936,0.566486,2.222767,1.0,1.0,0.0,1.0,0.0


**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [30]:
fraud['fraud'].value_counts(normalize=True)

fraud
0.0    0.912597
1.0    0.087403
Name: proportion, dtype: float64

In [31]:
fraud.dtypes

distance_from_home                float64
distance_from_last_transaction    float64
ratio_to_median_purchase_price    float64
repeat_retailer                   float64
used_chip                         float64
used_pin_number                   float64
online_order                      float64
fraud                             float64
dtype: object

In [ ]:
features_df = fraud.drop(columns=['fraud'])
target_df = fraud['fraud']
X_train,X_test,y_train,y_test = train_test_split(features_df,target_df,test_size= 0.2,random_state=42)
X_train.shape,X_test.shape

y_train.value_counts(normalize=True)


((800000, 7), (200000, 7))

#### built  logistics regression model with above inbalanced data

In [33]:
from sklearn.linear_model import LogisticRegression


log_model = LogisticRegression(
                class_weight="balanced",
                penalty="l2",
                max_iter=500,

)

log_model.fit(X_train,y_train)

prediction = log_model.predict(X_test)

In [ ]:
# focus on class fraud 
# when the model predicts fraud, only 58% of those cases are actually fraud, 42% are false alarms(non-fraud incorrectly flagged as fraud)
# the system will waste resources reviewing many false alarms, which can be costly and frustrating.

print("PR AUC:", average_precision_score(y_test, prediction))

print(f"confusion matrix:","\n", confusion_matrix(y_test,prediction))

print(f"Classification Report:","\n", classification_report(y_test,prediction))

# Effect of imbalance on precision 
'''
In an imbalanced dataset, the majority class (non-fraud = 91%) dominates.

Even a small false positive rate on the majority class generates a large number of false positives (because there are so many non-fraud cases).

These false positives directly appear in the denominator of precision (TP+FP), reducing precision.
'''


PR AUC: 0.5523661608077185
confusion matrix: 
 [[170348  12209]
 [   848  16595]]
Classification Report: 
               precision    recall  f1-score   support

         0.0       1.00      0.93      0.96    182557
         1.0       0.58      0.95      0.72     17443

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



#### oversample trainning data

In [44]:
over_sampler = RandomOverSampler()

X_train_over,y_train_over = over_sampler.fit_resample(X_train,y_train)

pd.Series(y_train_over).value_counts()

log_model_over = LogisticRegression(max_iter=2000)

log_model_over.fit(X_train_over,y_train_over)

pred_over = log_model_over.predict(X_test)

print(f"Classification Report:","\n", classification_report(y_test,pred_over))



Classification Report: 
               precision    recall  f1-score   support

         0.0       1.00      0.93      0.96    182557
         1.0       0.58      0.95      0.72     17443

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



#### undersample train data

In [45]:


under_sampler = RandomUnderSampler()

X_train_under,y_train_under = under_sampler.fit_resample(X_train,y_train)

pd.Series(y_train_under).value_counts()

log_model_under = LogisticRegression(max_iter=2000)

log_model_under.fit(X_train_under,y_train_under)

pred_under = log_model_under.predict(X_test)

print(f"Classification Report:","\n", classification_report(y_test,pred_under))


Classification Report: 
               precision    recall  f1-score   support

         0.0       1.00      0.93      0.96    182557
         1.0       0.58      0.95      0.72     17443

    accuracy                           0.93    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.93      0.94    200000



#### SMOTE 

In [46]:
smooth_sampler = SMOTE()

X_train_smooth,y_train_smooth = smooth_sampler.fit_resample(X_train,y_train)

pd.Series(y_train_smooth).value_counts()

log_reg_smooth = LogisticRegression(max_iter=2000)
log_reg_smooth.fit(X_train_smooth,y_train_smooth)

pred_smooth = log_reg_smooth.predict(X_test)

print(f"Classification Report:","\n", classification_report(y_test,pred_smooth))


Classification Report: 
               precision    recall  f1-score   support

         0.0       0.99      0.93      0.96    182557
         1.0       0.58      0.95      0.72     17443

    accuracy                           0.94    200000
   macro avg       0.79      0.94      0.84    200000
weighted avg       0.96      0.94      0.94    200000

